In [3]:
import os


In [4]:
%cd ..

c:\Users\badsh\OneDrive\Desktop\chicken-disease-classification


In [19]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL : str
    local_data_file : Path
    unzip_dir: Path

In [20]:
from cnnClassifier.constants import *
from cnnClassifier.utils.comman import read_yaml, create_directories


In [27]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [24]:
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.comman import get_size

In [32]:
class DataIngestion:

    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info:\n{headers}")
        else:
            logger.info(
                f"File already exist of size: "
                f"{get_size(Path(self.config.local_data_file))}"
            )

    def extract_zip_file(self):
        """
        Extract the zip file into the directory.
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(
            self.config.local_data_file, 'r'
        ) as zip_ref:
            zip_ref.extractall(unzip_path)


In [33]:
from pathlib import Path

try:
    config = ConfigurationManager(
        config_filepath=Path("config/config.yaml"),
        params_filepath=Path("params.yml")
    )

    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(
        config=data_ingestion_config
    )

    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-10-01 14:41:22,592: INFO:comman:yaml file: config\config.yaml loaded sucessfully]
[2026-10-01 14:41:22,595: INFO:comman:yaml file: params.yml loaded sucessfully]
[2026-10-01 14:41:22,600: INFO:comman:created directory at:artifacts]
[2026-10-01 14:41:22,603: INFO:comman:created directory at:artifacts/data_ingestion]
[2026-10-01 14:42:42,411: INFO:1487590676:artifacts/data_ingestion/data.zip downloaded! with following info:
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 5BE2:2E99AE:12EEA7:29669B:6ABE23BD
x-github-edge-region: centralindia
Accept-Ranges: bytes
Date: Thu, 01 Oct 2026 09:11:26 GMT
Via: 1.1 varnish
X-Served-By: cache-b